# BUS 104: Switching Costs, Solvers, and Sensitivity (`PuLP 4.0`)
**Case Study:** Suncrest Artisan Lighting — Incremental Cell-by-Cell Teaching Notebook

---
## Part 1: Product Mix with Switching Costs (MILP — Slides 4–21)

### Step 1.1: Import `PuLP 4.0`

In [96]:
import pulp as pl

### Step 1.2: Initialize the MILP Maximization Model Container

In [97]:
prob = pl.LpProblem("Suncrest_MILP", pl.LpMaximize)

### Step 1.3: Inspect the Empty Model Container (`print(prob)`)

In [98]:
print(prob)

Suncrest_MILP:
MAXIMIZE
None
VARIABLES



### Step 2.1: Add Continuous Production Variable `x1` (Desk Lamps $\ge 0$)

In [99]:
x1 = prob.add_variable("x1", lowBound=0, cat="Continuous")

### Step 2.2: Add Continuous Production Variable `x2` (Floor Lamps $\ge 0$)

In [100]:
x2 = prob.add_variable("x2", lowBound=0, cat="Continuous")

### Step 2.3: Add Binary Switch Variable `y1` (Desk Line Setup $\in \{0, 1\}$)

In [101]:
y1 = prob.add_variable("y1", cat="Binary")

### Step 2.4: Add Binary Switch Variable `y2` (Floor Line Setup $\in \{0, 1\}$)

In [102]:
y2 = prob.add_variable("y2", cat="Binary")

### Step 3.1: Add the Net Profit Objective ($50x_1 + 30x_2 - 80y_1 - 60y_2$)

In [103]:
prob += 50 * x1 + 30 * x2 - 80 * y1 - 60 * y2, "Net_Profit"

### Step 3.2: Inspect the Model After Adding the Objective

In [104]:
print(prob)

Suncrest_MILP:
MAXIMIZE
50.0*x1 + 30.0*x2 + -80.0*y1 + -60.0*y2 + 0.0
VARIABLES
x1 Continuous
x2 Continuous
0 <= y1 <= 1 Integer
0 <= y2 <= 1 Integer



### Step 3.3: Add the Assembly Department Constraint ($x_1 + x_2 \le 10$)

In [105]:
prob += 1 * x1 + 1 * x2 <= 10, "Assembly"

### Step 3.4: Add the Machining Department Constraint ($2x_1 + x_2 \le 16$)

In [106]:
prob += 2 * x1 + 1 * x2 <= 16, "Machining"

### Step 3.5: Add the Finishing Department Constraint ($x_2 \le 7$)

In [107]:
prob += 1 * x2 <= 7, "Finishing"

### Step 3.6: Inspect the Model After Adding the 3 Resource Constraints

In [108]:
print(prob)

Suncrest_MILP:
MAXIMIZE
50.0*x1 + 30.0*x2 + -80.0*y1 + -60.0*y2 + 0.0
SUBJECT TO
Assembly: x1 + x2 <= 10

Machining: 2 x1 + x2 <= 16

Finishing: x2 <= 7

VARIABLES
x1 Continuous
x2 Continuous
0 <= y1 <= 1 Integer
0 <= y2 <= 1 Integer



### Step 3.7: Add the Desk Lamp Big-$M$ Linking Constraint ($x_1 - 8y_1 \le 0$, $M_1 = 8$)

In [109]:
prob += x1 - 8 * y1 <= 0, "Link_Desk"

### Step 3.8: Add the Floor Lamp Big-$M$ Linking Constraint ($x_2 - 7y_2 \le 0$, $M_2 = 7$)

In [110]:
prob += x2 - 7 * y2 <= 0, "Link_Floor"

### Step 3.9: Inspect the Complete 5-Constraint MILP Formulation Before Solving

In [111]:
print(prob)

Suncrest_MILP:
MAXIMIZE
50.0*x1 + 30.0*x2 + -80.0*y1 + -60.0*y2 + 0.0
SUBJECT TO
Assembly: x1 + x2 <= 10

Machining: 2 x1 + x2 <= 16

Finishing: x2 <= 7

Link_Desk: x1 - 8 y1 <= 0

Link_Floor: x2 - 7 y2 <= 0

VARIABLES
x1 Continuous
x2 Continuous
0 <= y1 <= 1 Integer
0 <= y2 <= 1 Integer



### Step 4.1: Solve the MILP Model in Colab requires a CBC solver installation

In [112]:
!apt-get install -y coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.11+ds1-2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [113]:
cbc_solver = pl.COIN_CMD(path='/usr/bin/cbc', msg=False)

In [114]:
# Solve by explicitly directing PuLP to the newly installed system CBC solver executable
res = prob.solve(cbc_solver)

### Step 4.2: Print the Solver Status

In [115]:
print("Status:", res.status.name)

Status: Optimal


### Step 4.3: Print the Optimal Net Profit

In [116]:
print("Optimal Profit =", pl.value(prob.objective))

Optimal Profit = 320.0


### Step 4.4: Print Optimal Desk Lamps (`x1.varValue`)

In [117]:
print("x1 =", x1.varValue)

x1 = 8.0


### Step 4.5: Print Optimal Floor Lamps (`x2.varValue`)

In [118]:
print("x2 =", x2.varValue)

x2 = 0.0


### Step 4.6: Print Desk Line Switch (`y1.varValue`)

In [119]:
print("y1 =", y1.varValue)

y1 = 1.0


### Step 4.7: Print Floor Line Switch (`y2.varValue`)

In [120]:
print("y2 =", y2.varValue)

y2 = 0.0


### Step 4.8: Print All Decision Variables Using a Simple 2-Line `for` Loop

In [121]:
for v in prob.variables():
    print(v.name, "Value =", v.varValue)

x1 Value = 8.0
x2 Value = 0.0
y1 Value = 1.0
y2 Value = 0.0


### Step 4.9: Live Scenario — You Make the Call #1 ($F_2 = \$15 \implies Z^* = \$325$ at $(6,4)$)

In [122]:
prob_ex1 = pl.LpProblem("Suncrest_MILP_F2_15", pl.LpMaximize)
x1_e = prob_ex1.add_variable("x1", lowBound=0, cat="Continuous")
x2_e = prob_ex1.add_variable("x2", lowBound=0, cat="Continuous")
y1_e = prob_ex1.add_variable("y1", cat="Binary")
y2_e = prob_ex1.add_variable("y2", cat="Binary")
prob_ex1 += 50 * x1_e + 30 * x2_e - 80 * y1_e - 15 * y2_e, "Net_Profit"
prob_ex1 += 1 * x1_e + 1 * x2_e <= 10, "Assembly"
prob_ex1 += 2 * x1_e + 1 * x2_e <= 16, "Machining"
prob_ex1 += 1 * x2_e <= 7, "Finishing"
prob_ex1 += x1_e - 8 * y1_e <= 0, "Link_Desk"
prob_ex1 += x2_e - 7 * y2_e <= 0, "Link_Floor"
res_ex1 = prob_ex1.solve(cbc_solver)
print("Status:", res_ex1.status.name)
print("Optimal Profit =", pl.value(prob_ex1.objective))
for v in prob_ex1.variables():
    print(v.name, "Value =", v.varValue)

Status: Optimal
Optimal Profit = 325.0
x1 Value = 6.0
x2 Value = 4.0
y1 Value = 1.0
y2 Value = 1.0


---
## Part 2: Removing Switching Costs $\rightarrow$ Slack, Shadow Price, & Reduced Cost (Slides 22–31)

### Step 5.1 (Slide 23): Initialize the 2-Product Continuous LP (`lp`)

In [123]:
lp = pl.LpProblem("Suncrest_LP", pl.LpMaximize)

### Step 5.2 (Slide 23): Add Continuous Variable `x1` (Desk Lamps)

In [124]:
x1 = lp.add_variable("x1", lowBound=0, cat="Continuous")

### Step 5.3 (Slide 23): Add Continuous Variable `x2` (Floor Lamps)

In [125]:
x2 = lp.add_variable("x2", lowBound=0, cat="Continuous")

### Step 5.4 (Slide 23): Add the Profit Objective ($50x_1 + 30x_2$)

In [126]:
lp += 50 * x1 + 30 * x2, "Profit"

### Step 5.5 (Slide 23): Add Assembly Constraint ($x_1 + x_2 \le 10$)

In [127]:
lp += 1 * x1 + 1 * x2 <= 10, "Assembly"

### Step 5.6 (Slide 23): Add Machining Constraint ($2x_1 + x_2 \le 16$)

In [128]:
lp += 2 * x1 + 1 * x2 <= 16, "Machining"

### Step 5.7 (Slide 23): Add Finishing Constraint ($x_2 \le 7$)

In [129]:
lp += 1 * x2 <= 7, "Finishing"

### Step 5.8 (Slide 23): Inspect the 2-Product Continuous LP

In [130]:
print(lp)

Suncrest_LP:
MAXIMIZE
50.0*x1 + 30.0*x2 + 0.0
SUBJECT TO
Assembly: x1 + x2 <= 10

Machining: 2 x1 + x2 <= 16

Finishing: x2 <= 7

VARIABLES
x1 Continuous
x2 Continuous



### Step 5.9 (Slide 23): Solve the 2-Product Continuous LP and Print Profit

In [131]:
res = lp.solve(cbc_solver)
print("Profit =", pl.value(lp.objective))

Profit = 420.0


### Step 5.10 (Slide 23): Print Optimal Quantities `x1` and `x2`

In [132]:
print("x1 =", x1.varValue, "x2 =", x2.varValue)

x1 = 6.0 x2 = 4.0


### Step 5.11 (Slides 23–25): Print Shadow Price (`c.pi`) and Slack (`c.slack`) for Each Department

In [133]:
for c in lp.constraints():
    print(c.name, "Shadow Price =", c.pi, "Slack =", c.slack)

Assembly Shadow Price = -10.0 Slack = -0.0
Machining Shadow Price = -20.0 Slack = -0.0
Finishing Shadow Price = 0.0 Slack = 3.0


### Step 5.12 (Slide 26): How Is a Shadow Price Computed? Test $+1$ Hour of Machining (`16` $\rightarrow$ `17`)

In [134]:
lp_plus1 = pl.LpProblem("Suncrest_Machining_17", pl.LpMaximize)
x1_p = lp_plus1.add_variable("x1", lowBound=0, cat="Continuous")
x2_p = lp_plus1.add_variable("x2", lowBound=0, cat="Continuous")
lp_plus1 += 50 * x1_p + 30 * x2_p, "Profit"
lp_plus1 += 1 * x1_p + 1 * x2_p <= 10, "Assembly"
lp_plus1 += 2 * x1_p + 1 * x2_p <= 17, "Machining"
lp_plus1 += 1 * x2_p <= 7, "Finishing"
lp_plus1.solve(cbc_solver)
print("New Profit =", pl.value(lp_plus1.objective))
print("x1 =", x1_p.varValue, "x2 =", x2_p.varValue)

New Profit = 440.0
x1 = 7.0 x2 = 3.0


---
### Step 6.1 (Slide 29): Computing Reduced Cost in Python — Add 3rd Product `x3` (Pendant Lamp at $\$25$ Profit)

In [136]:
lp3 = pl.LpProblem("Suncrest_3_Products", pl.LpMaximize)
x1 = lp3.add_variable("x1", lowBound=0, cat="Continuous")
x2 = lp3.add_variable("x2", lowBound=0, cat="Continuous")
x3 = lp3.add_variable("x3", lowBound=0, cat="Continuous")
lp3 += 50 * x1 + 30 * x2 + 25 * x3, "Profit"
lp3 += 1 * x1 + 1 * x2 + 1 * x3 <= 10, "Assembly"
lp3 += 2 * x1 + 1 * x2 + 1 * x3 <= 16, "Machining"
lp3 += 1 * x2 + 1 * x3 <= 7, "Finishing"
lp3.solve(cbc_solver)

LpSolveStats(solver='COIN_CMD', status=<LpSolveStatus.Optimal: 1>, has_solution=True, time=0.014982938766479492, cpu_time=0.010000000000001563, objective=420.0, best_bound=None, num_variables=3, num_constraints=3, is_mip=False, solver_options={'solver': 'COIN_CMD', 'mip': True, 'msg': False, 'keepFiles': False, 'warmStart': False, 'timeMode': 'elapsed', 'logPath': '/tmp/tmptsrqu0nj-pulp.log'}, logs={'version': '2.10.11', 'solver': 'CBC', 'status': None, 'best_bound': None, 'best_solution': None, 'gap': None, 'time': 0.0, 'matrix_post': None, 'matrix': {'constraints': 3, 'variables': 3, 'nonzeros': 8}, 'cut_info': None, 'rootTime': None, 'presolve': None, 'first_relaxed': None, 'progress': [], 'first_solution': None, 'status_code': None, 'sol_code': None, 'nodes': None})

### Step 6.2 (Slide 29): Print `v.varValue` and `v.dj` (Reduced Cost) for All 3 Products

In [137]:
for v in lp3.variables():
    print(v.name, "Value =", v.varValue, "Reduced Cost =", v.dj)

x1 Value = 6.0 Reduced Cost = 0.0
x2 Value = 4.0 Reduced Cost = 0.0
x3 Value = 0.0 Reduced Cost = 5.0


### Step 6.3 (Slide 30): Live Test — Raise Pendant Lamp (`x3`) Profit from $\$25$ to $\$35$ (Above $\$30$!)

In [138]:
lp3_35 = pl.LpProblem("Suncrest_x3_at_35", pl.LpMaximize)
x1 = lp3_35.add_variable("x1", lowBound=0, cat="Continuous")
x2 = lp3_35.add_variable("x2", lowBound=0, cat="Continuous")
x3 = lp3_35.add_variable("x3", lowBound=0, cat="Continuous")
lp3_35 += 50 * x1 + 30 * x2 + 35 * x3, "Profit"
lp3_35 += 1 * x1 + 1 * x2 + 1 * x3 <= 10, "Assembly"
lp3_35 += 2 * x1 + 1 * x2 + 1 * x3 <= 16, "Machining"
lp3_35 += 1 * x2 + 1 * x3 <= 7, "Finishing"
lp3_35.solve(cbc_solver)
print("New Profit =", pl.value(lp3_35.objective))
for v in lp3_35.variables():
    print(v.name, "Value =", v.varValue, "Reduced Cost =", v.dj)

New Profit = 440.0
x1 Value = 6.0 Reduced Cost = 0.0
x2 Value = 0.0 Reduced Cost = 5.0
x3 Value = 4.0 Reduced Cost = 0.0
